# SQL for Data Engineers and Data Scientists
## From Fundamentals to Advanced Mastery

This notebook covers SQL comprehensively using SQLite via Python's built-in `sqlite3` module.
All examples run without any external database installation.

---

## Table of Contents

1. Introduction to Databases and SQL
2. SELECT — Basic Queries
3. Filtering and Sorting
4. Aggregation Functions
5. JOINs
6. Subqueries and CTEs
7. Window Functions
8. Data Manipulation Language (DML)
9. DDL and Schema Design
10. Python SQL Integration (sqlite3, pandas, SQLAlchemy)

---


# Section 1 — Introduction to Databases and SQL

## Concept

A **relational database** stores data in tables (relations) made up of rows and columns.
Tables relate to each other through keys. SQL (Structured Query Language) is the standard
language used to query and manipulate relational data.

**Core terminology:**

| Term | Description |
|---|---|
| Table | A named collection of rows with a fixed set of columns |
| Row / Record | A single data entry in a table |
| Column / Field | An attribute of the data |
| Primary Key | Uniquely identifies each row |
| Foreign Key | References the primary key of another table |
| Schema | The structure definition of a database |
| Index | A data structure that speeds up lookups |
| Transaction | An atomic unit of work (all or nothing) |

## Technical Deep Dive

SQL is divided into sublanguages:

- **DDL** — Data Definition Language: `CREATE`, `ALTER`, `DROP`
- **DML** — Data Manipulation Language: `INSERT`, `UPDATE`, `DELETE`
- **DQL** — Data Query Language: `SELECT`
- **DCL** — Data Control Language: `GRANT`, `REVOKE`
- **TCL** — Transaction Control Language: `COMMIT`, `ROLLBACK`, `SAVEPOINT`

Popular RDBMS: **PostgreSQL**, **MySQL**, **SQLite**, **SQL Server**, **Oracle**.
SQLite is embedded, file-based, and ideal for learning and small applications.


In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database (reusable across all sections)
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row  # enables column access by name
cursor = conn.cursor()

# Enable foreign keys
cursor.execute('PRAGMA foreign_keys = ON')

# Helper function to display query results as DataFrame
def q(sql, params=None):
    if params:
        return pd.read_sql_query(sql, conn, params=params)
    return pd.read_sql_query(sql, conn)

print('SQLite version:', sqlite3.sqlite_version)
print('Connection ready.')

In [ ]:
# Create and populate sample database used throughout this course

cursor.executescript('''
CREATE TABLE departments (
    id       INTEGER PRIMARY KEY,
    name     TEXT NOT NULL UNIQUE,
    budget   REAL,
    location TEXT
);

CREATE TABLE employees (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    department  TEXT,
    salary      REAL,
    hire_date   TEXT,
    manager_id  INTEGER REFERENCES employees(id)
);

CREATE TABLE projects (
    id            INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    department_id INTEGER REFERENCES departments(id),
    start_date    TEXT,
    end_date      TEXT,
    budget        REAL
);

CREATE TABLE employee_projects (
    employee_id INTEGER REFERENCES employees(id),
    project_id  INTEGER REFERENCES projects(id),
    role        TEXT,
    PRIMARY KEY (employee_id, project_id)
);

INSERT INTO departments VALUES
    (1, 'Engineering', 500000, 'New York'),
    (2, 'Marketing',   200000, 'Chicago'),
    (3, 'Sales',       300000, 'Los Angeles'),
    (4, 'HR',          150000, 'New York'),
    (5, 'Finance',     250000, 'Chicago');

INSERT INTO employees VALUES
    (1,  'Alice Johnson',  'Engineering', 95000, '2019-03-15', NULL),
    (2,  'Bob Smith',      'Engineering', 85000, '2020-06-01', 1),
    (3,  'Carol White',    'Marketing',   75000, '2018-11-20', NULL),
    (4,  'David Lee',      'Sales',       65000, '2021-01-10', NULL),
    (5,  'Eve Davis',      'Engineering', 90000, '2019-07-22', 1),
    (6,  'Frank Wilson',   'HR',          60000, '2022-03-05', NULL),
    (7,  'Grace Taylor',   'Finance',     80000, '2020-09-14', NULL),
    (8,  'Henry Brown',    'Sales',       70000, '2021-05-18', 4),
    (9,  'Iris Martinez',  'Marketing',   72000, '2019-12-01', 3),
    (10, 'Jack Anderson',  'Engineering', 88000, '2020-02-28', 1);

INSERT INTO projects VALUES
    (1, 'Platform Rewrite',      1, '2023-01-01', '2023-12-31', 150000),
    (2, 'Marketing Campaign Q1', 2, '2023-01-15', '2023-03-31',  50000),
    (3, 'Sales Automation',      3, '2023-02-01', '2023-08-31',  75000),
    (4, 'HR System Upgrade',     4, '2023-03-01', '2023-06-30',  30000),
    (5, 'Data Pipeline',         1, '2023-04-01', '2023-10-31', 120000);

INSERT INTO employee_projects VALUES
    (1, 1, 'Lead'), (2, 1, 'Developer'), (5, 1, 'Developer'),
    (10,1, 'Developer'), (3, 2, 'Lead'), (9, 2, 'Analyst'),
    (4, 3, 'Lead'), (8, 3, 'Developer'), (6, 4, 'Lead'),
    (1, 5, 'Lead'), (2, 5, 'Developer'), (7, 5, 'Analyst');
''')
conn.commit()
print('Sample database ready.')

## Summary

- Relational databases store data in tables connected via keys.
- SQL is split into DDL, DML, DQL, DCL, TCL.
- SQLite runs in-process — no server required.
- We created 4 tables: `departments`, `employees`, `projects`, `employee_projects`.

---


# Section 2 — SELECT: Basic Queries

## Concept

`SELECT` retrieves data from one or more tables. It is the most common SQL statement.

Basic syntax:
```sql
SELECT column1, column2
FROM   table_name;
```

## Technical Deep Dive

| Clause | Purpose |
|---|---|
| `SELECT *` | All columns |
| `SELECT col AS alias` | Column with alias |
| `DISTINCT` | Remove duplicate rows |
| `LIMIT n` | Return at most n rows |
| `OFFSET n` | Skip n rows (pagination) |
| Expressions | `salary * 1.1`, `UPPER(name)` etc. |

**Execution order** (not the same as write order):
```
FROM → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT
```


In [ ]:
# Select all columns
print('=== All employees ===')
print(q('SELECT * FROM employees'))

# Select specific columns with alias
print('\n=== Name and salary ===')
print(q('SELECT name, salary, department AS dept FROM employees'))

# Computed column
print('\n=== Salary with 10% raise ===')
print(q('SELECT name, salary, ROUND(salary * 1.1, 2) AS salary_raised FROM employees'))

In [ ]:
# DISTINCT — unique departments
print('=== Distinct departments ===')
print(q('SELECT DISTINCT department FROM employees ORDER BY department'))

# LIMIT and OFFSET — pagination
print('\n=== Top 3 employees (page 1) ===')
print(q('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3'))

print('\n=== Next 3 employees (page 2) ===')
print(q('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3 OFFSET 3'))

## Exercises

1. Select only `name` and `hire_date` from `employees`, ordered by `hire_date` ascending.
2. List all distinct locations from `departments`.
3. Select the 5th and 6th employees by `id`.
4. Show each employee's name and their annual salary expressed as monthly (`salary / 12`), aliased `monthly_salary`.

## Mini Challenge

Write a query that shows the 3 longest-serving employees (earliest `hire_date`), displaying `name`, `department`, `hire_date`, and the number of full years they have worked (use SQLite's `CAST` and date functions).

## Best Practices

- Always list columns explicitly instead of `SELECT *` in production code.
- Use meaningful aliases for computed columns.
- Always pair `LIMIT` with `ORDER BY` — without ORDER BY the result order is undefined.

## Common Mistakes

- Using `SELECT *` in production — returns hidden columns and breaks when schema changes.
- Forgetting `ORDER BY` with `LIMIT` — results are non-deterministic.
- Confusing `OFFSET` semantics — `OFFSET 0` means no skip.

## Performance Notes

- `SELECT *` forces the database to read all columns — unnecessary I/O.
- `LIMIT` is applied after sorting; sorting still scans all rows unless indexed.

## Summary

- `SELECT` retrieves columns; use aliases for clarity.
- `DISTINCT` deduplicates rows.
- `LIMIT` + `OFFSET` enables pagination.
- SQL execution order differs from write order.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
# Exercise 1: name and hire_date ordered by hire_date
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2: distinct locations
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3: 5th and 6th employee by id
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4: monthly salary
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: 3 longest-serving employees with years worked
# YOUR SQL HERE
print(q(''))

# Section 3 — Filtering and Sorting

## Concept

`WHERE` filters rows before aggregation. `ORDER BY` sorts the result set.

## Technical Deep Dive

**Comparison operators:** `=`, `<>`, `!=`, `<`, `>`, `<=`, `>=`

**Logical operators:** `AND`, `OR`, `NOT`

**Special predicates:**

| Predicate | Example |
|---|---|
| `BETWEEN a AND b` | `salary BETWEEN 70000 AND 90000` |
| `IN (list)` | `department IN ('Sales', 'HR')` |
| `LIKE pattern` | `name LIKE 'A%'` |
| `IS NULL` | `manager_id IS NULL` |
| `IS NOT NULL` | `manager_id IS NOT NULL` |

**LIKE wildcards:** `%` matches any sequence, `_` matches exactly one character.

**ORDER BY:** `ASC` (default) or `DESC`. Can sort by multiple columns:
```sql
ORDER BY department ASC, salary DESC
```

**NULL handling:** `NULL` comparisons always evaluate to UNKNOWN (not TRUE/FALSE).
Use `IS NULL` / `IS NOT NULL`, never `= NULL`.


In [ ]:
# Basic WHERE
print('=== Engineering employees ===')
print(q("SELECT name, salary FROM employees WHERE department = 'Engineering'"))

# BETWEEN
print('\n=== Salary between 70000 and 90000 ===')
print(q('SELECT name, salary FROM employees WHERE salary BETWEEN 70000 AND 90000'))

# IN
print("\n=== Sales or HR employees ===")
print(q("SELECT name, department FROM employees WHERE department IN ('Sales', 'HR')"))

In [ ]:
# LIKE
print('=== Names starting with A or B ===')
print(q("SELECT name FROM employees WHERE name LIKE 'A%' OR name LIKE 'B%'"))

# IS NULL — top-level managers (no manager_id)
print('\n=== Top-level managers (no manager) ===')
print(q('SELECT name, department FROM employees WHERE manager_id IS NULL'))

# Combined conditions
print('\n=== Engineering employees hired before 2020 ===')
print(q("SELECT name, hire_date FROM employees WHERE department = 'Engineering' AND hire_date < '2020-01-01'"))

In [ ]:
# Multi-column ORDER BY
print('=== Employees ordered by dept ASC, salary DESC ===')
print(q('SELECT name, department, salary FROM employees ORDER BY department ASC, salary DESC'))

## Exercises

1. Find all employees with salary greater than 85000.
2. Find employees hired between 2020-01-01 and 2021-12-31.
3. Find employees whose name contains the letter 'a' (case-insensitive).
4. List employees who are NOT in Engineering and have a manager.

## Mini Challenge

Write a query that returns employees earning above the median salary (hardcode 80000 as median approximation).
Show `name`, `department`, `salary`, and a column `salary_tier` that is `'High'` if salary > 85000, `'Mid'` if 70000-85000, `'Low'` otherwise. Use a `CASE` expression.

## Best Practices

- Use `BETWEEN` for range checks — more readable than `>= AND <=`.
- Avoid `NOT IN` with subqueries that might return NULL — use `NOT EXISTS` instead.
- Index columns used in `WHERE` and `ORDER BY` for large tables.

## Common Mistakes

- `WHERE col = NULL` never matches anything — use `IS NULL`.
- `LIKE '%word%'` cannot use an index — avoid leading wildcards on large tables.
- `NOT IN (subquery with NULLs)` returns no rows — a silent gotcha.

## Summary

- `WHERE` filters rows using comparison, logical, and special predicates.
- `BETWEEN`, `IN`, `LIKE`, `IS NULL` are the key predicates.
- `CASE` expressions produce conditional columns inline.
- NULL requires special handling — never use `= NULL`.

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3 — case-insensitive LIKE in SQLite
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: salary tier using CASE
# YOUR SQL HERE
print(q(''))

# Section 4 — Aggregation Functions

## Concept

Aggregate functions collapse multiple rows into a single value.
`GROUP BY` divides rows into groups before aggregation.
`HAVING` filters groups after aggregation.

## Technical Deep Dive

| Function | Description |
|---|---|
| `COUNT(*)` | Count all rows |
| `COUNT(col)` | Count non-NULL values |
| `COUNT(DISTINCT col)` | Count unique non-NULL values |
| `SUM(col)` | Sum of values |
| `AVG(col)` | Arithmetic mean |
| `MIN(col)` | Minimum value |
| `MAX(col)` | Maximum value |
| `GROUP_CONCAT(col)` | Concatenate values (SQLite) |

**Rule:** Every column in `SELECT` must either be in `GROUP BY` or wrapped in an aggregate function.

**WHERE vs HAVING:**
- `WHERE` filters rows *before* aggregation.
- `HAVING` filters groups *after* aggregation.


In [ ]:
# Basic aggregates
print('=== Global stats ===')
print(q('SELECT COUNT(*) AS total, AVG(salary) AS avg_salary, MIN(salary) AS min_sal, MAX(salary) AS max_sal FROM employees'))

# GROUP BY
print('\n=== Stats per department ===')
print(q('''
    SELECT department,
           COUNT(*)        AS headcount,
           ROUND(AVG(salary), 0) AS avg_salary,
           SUM(salary)     AS total_payroll
    FROM employees
    GROUP BY department
    ORDER BY total_payroll DESC
'''))

In [ ]:
# HAVING — filter groups
print('=== Departments with avg salary > 75000 ===')
print(q('''
    SELECT department, ROUND(AVG(salary), 0) AS avg_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 75000
    ORDER BY avg_salary DESC
'''))

# GROUP_CONCAT
print('\n=== Employees per department (names) ===')
print(q('''
    SELECT department, GROUP_CONCAT(name, ', ') AS members
    FROM employees
    GROUP BY department
    ORDER BY department
'''))

## Exercises

1. Count the number of employees per department, ordered by count descending.
2. Find the department with the highest average salary.
3. Find all departments that have more than 2 employees.
4. Calculate the total budget of all projects.

## Mini Challenge

Create a salary distribution report showing: `department`, `headcount`, `avg_salary`, `min_salary`, `max_salary`, `salary_range` (max - min). Include only departments where headcount >= 2, ordered by `avg_salary` descending.

## Best Practices

- Use `COUNT(*)` to count rows; `COUNT(col)` to count non-NULL values of a column.
- Filter with `WHERE` before grouping when possible — it reduces the data aggregated.
- Always use `GROUP BY` before `HAVING` in the query clause order.

## Common Mistakes

- Selecting a non-aggregated column not in `GROUP BY` — undefined behavior in strict mode.
- Using `WHERE` instead of `HAVING` to filter aggregated values.
- `AVG` ignores NULLs; `COUNT(*)` counts NULLs; `COUNT(col)` does not.

## Summary

- Aggregate functions summarize data: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.
- `GROUP BY` groups rows; `HAVING` filters those groups.
- Execution order: `WHERE` → `GROUP BY` → `HAVING` → `SELECT`.

---


### Exercise and Challenge Solutions — Section 4


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge
# YOUR SQL HERE
print(q(''))

# Section 5 — JOINs

## Concept

JOINs combine rows from two or more tables based on a related column.

## Technical Deep Dive

| JOIN Type | Description |
|---|---|
| `INNER JOIN` | Rows with matching values in both tables |
| `LEFT JOIN` | All rows from left table; NULLs for non-matches on right |
| `RIGHT JOIN` | All rows from right (SQLite: emulate via reversed LEFT JOIN) |
| `FULL OUTER JOIN` | All rows from both tables (emulate in SQLite) |
| `CROSS JOIN` | Cartesian product — every row × every row |
| `SELF JOIN` | Table joined to itself |

```sql
SELECT a.col, b.col
FROM   table_a AS a
JOIN   table_b AS b ON a.id = b.a_id;
```

**Anti-join pattern** — rows in A with no match in B:
```sql
SELECT * FROM a LEFT JOIN b ON a.id = b.a_id WHERE b.id IS NULL;
```


In [ ]:
# INNER JOIN — employees with their projects
print('=== Employees and their project roles ===')
sql = '''
SELECT e.name, p.name AS project, ep.role
FROM employees e
JOIN employee_projects ep ON e.id = ep.employee_id
JOIN projects p           ON ep.project_id = p.id
ORDER BY e.name, p.name
'''
print(q(sql))

In [ ]:
# LEFT JOIN — all employees, even those without projects
print('=== All employees (including without projects) ===')
sql = '''
SELECT e.name, p.name AS project
FROM employees e
LEFT JOIN employee_projects ep ON e.id = ep.employee_id
LEFT JOIN projects p           ON ep.project_id = p.id
ORDER BY e.name
'''
print(q(sql))

# Anti-join — employees with NO projects
print('\n=== Employees with no project assignments ===')
sql = '''
SELECT e.name
FROM employees e
LEFT JOIN employee_projects ep ON e.id = ep.employee_id
WHERE ep.employee_id IS NULL
'''
print(q(sql))

In [ ]:
# SELF JOIN — employee with their manager
print('=== Employee - Manager pairs ===')
sql = '''
SELECT e.name AS employee, m.name AS manager
FROM employees e
JOIN employees m ON e.manager_id = m.id
ORDER BY manager, employee
'''
print(q(sql))

In [ ]:
# FULL OUTER JOIN emulation in SQLite (LEFT JOIN UNION ALL anti-right-join)
print('=== Departments with their projects (FULL OUTER JOIN emulation) ===')
sql = '''
SELECT d.name AS department, p.name AS project
FROM departments d
LEFT JOIN projects p ON d.id = p.department_id

UNION ALL

SELECT d.name AS department, p.name AS project
FROM projects p
LEFT JOIN departments d ON p.department_id = d.id
WHERE d.id IS NULL
ORDER BY department
'''
print(q(sql))

## Exercises

1. List all projects with their department name.
2. Find all departments with no projects.
3. Show each employee with their manager's name (include employees with no manager).
4. Count how many projects each department has.

## Mini Challenge

Find the most expensive project per department. Show `department_name`, `project_name`, and `budget`. Use a JOIN and a subquery.

## Best Practices

- Always use table aliases for readability in multi-table queries.
- Prefer explicit `JOIN ... ON` over implicit comma-separated `FROM a, b WHERE`.
- Join on indexed columns for performance.

## Common Mistakes

- Forgetting the `ON` condition causes a CROSS JOIN (cartesian explosion).
- LEFT JOIN + WHERE on right-table column silently converts it to INNER JOIN.
- Joining on non-unique columns causes row multiplication.

## Summary

- INNER JOIN returns only matching rows.
- LEFT JOIN preserves all left-side rows.
- Anti-join pattern: LEFT JOIN + `WHERE right.id IS NULL`.
- SELF JOIN uses the same table twice with different aliases.

---


### Exercise and Challenge Solutions — Section 5


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2 — departments with no projects
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3 — employee with manager (include those without)
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: most expensive project per department
# YOUR SQL HERE
print(q(''))

# Section 6 — Subqueries and CTEs

## Concept

A **subquery** is a query nested inside another query.
A **CTE** (Common Table Expression) is a named temporary result set defined with `WITH`.

## Technical Deep Dive

**Subquery types:**

| Type | Location | Returns |
|---|---|---|
| Scalar | SELECT or WHERE | Single value |
| Row | WHERE | Single row |
| Table | FROM | Table (derived table) |
| Correlated | WHERE | One value per outer row |

**Operators used with subqueries:** `IN`, `NOT IN`, `EXISTS`, `NOT EXISTS`, `ANY`, `ALL`.

**CTE syntax:**
```sql
WITH cte_name AS (
    SELECT ...
)
SELECT * FROM cte_name;
```

**Recursive CTE:**
```sql
WITH RECURSIVE cte AS (
    SELECT ...         -- anchor
    UNION ALL
    SELECT ... FROM cte WHERE ...  -- recursive member
)
SELECT * FROM cte;
```


In [ ]:
# Scalar subquery — employees above average salary
print('=== Above average salary ===')
print(q('SELECT name, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees) ORDER BY salary DESC'))

# IN subquery — employees on project 1
print('\n=== Employees on Platform Rewrite ===')
print(q("SELECT name FROM employees WHERE id IN (SELECT employee_id FROM employee_projects WHERE project_id = 1)"))

In [ ]:
# EXISTS — employees who lead at least one project
print('=== Employees who are project leads ===')
sql = '''
SELECT name FROM employees e
WHERE EXISTS (
    SELECT 1 FROM employee_projects ep
    WHERE ep.employee_id = e.id AND ep.role = 'Lead'
)
ORDER BY name
'''
print(q(sql))

# Correlated subquery — salary vs department avg
print('\n=== Employee salary vs dept average ===')
sql = '''
SELECT
    name,
    department,
    salary,
    ROUND((SELECT AVG(e2.salary) FROM employees e2 WHERE e2.department = e1.department), 0) AS dept_avg
FROM employees e1
ORDER BY department, salary DESC
'''
print(q(sql))

In [ ]:
# CTE — department salary stats then filter
print('=== CTE example: dept stats with filter ===')
sql = '''
WITH dept_stats AS (
    SELECT
        department,
        COUNT(*)                    AS headcount,
        ROUND(AVG(salary), 0)       AS avg_salary,
        SUM(salary)                 AS total_salary
    FROM employees
    GROUP BY department
)
SELECT *
FROM dept_stats
WHERE avg_salary > 70000
ORDER BY avg_salary DESC
'''
print(q(sql))

In [ ]:
# Recursive CTE — org hierarchy starting from employee 2
print('=== Org hierarchy (recursive CTE) ===')
sql = '''
WITH RECURSIVE org_tree AS (
    -- Anchor: start from employee with no manager
    SELECT id, name, manager_id, 0 AS level
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive: find reports of current level
    SELECT e.id, e.name, e.manager_id, ot.level + 1
    FROM employees e
    JOIN org_tree ot ON e.manager_id = ot.id
)
SELECT level, name,
       (SELECT name FROM employees WHERE id = ot.manager_id) AS reports_to
FROM org_tree ot
ORDER BY level, name
'''
print(q(sql))

## Exercises

1. Using a subquery, find employees in the same department as 'Alice Johnson'.
2. Using `NOT EXISTS`, find employees not assigned to any project.
3. Write a CTE that ranks departments by total payroll, then select the top 2.
4. Find projects with a budget above the average project budget using a scalar subquery.

## Mini Challenge

Write a recursive CTE that generates numbers 1 to 10 (no base table needed). Then extend it to generate a Fibonacci sequence up to the 10th term.

## Best Practices

- Prefer CTEs over deeply nested subqueries for readability.
- Use `EXISTS` instead of `IN` with subqueries when the subquery is large.
- Add a `LIMIT` guard to recursive CTEs to prevent infinite loops during development.

## Common Mistakes

- `NOT IN` returns no rows when subquery contains NULL — use `NOT EXISTS` instead.
- Correlated subqueries run once per outer row — expensive on large tables.
- Missing `UNION ALL` in recursive CTEs (using `UNION` removes duplicates, breaking counters).

## Summary

- Subqueries can appear in SELECT, FROM, and WHERE clauses.
- CTEs improve readability and can be referenced multiple times.
- Recursive CTEs traverse hierarchical data.

---


### Exercise and Challenge Solutions — Section 6


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: generate 1-10 with recursive CTE
# YOUR SQL HERE
print(q(''))

# Section 7 — Window Functions

## Concept

Window functions perform calculations across a set of rows related to the current row,
without collapsing them into a single output row (unlike aggregate functions).

## Technical Deep Dive

```sql
function_name() OVER (
    PARTITION BY col     -- divide into groups
    ORDER BY col         -- order within group
    ROWS BETWEEN ...     -- frame specification
)
```

| Function | Description |
|---|---|
| `ROW_NUMBER()` | Sequential integer per partition |
| `RANK()` | Rank with gaps on ties |
| `DENSE_RANK()` | Rank without gaps on ties |
| `NTILE(n)` | Divide rows into n buckets |
| `LAG(col, n)` | Value n rows before current |
| `LEAD(col, n)` | Value n rows after current |
| `FIRST_VALUE(col)` | First value in the window |
| `LAST_VALUE(col)` | Last value in the window |
| `SUM() OVER (...)` | Running total |
| `AVG() OVER (...)` | Moving average |

**Frame specification:**
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — running total
- `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` — 3-row moving window


In [ ]:
# ROW_NUMBER, RANK, DENSE_RANK
print('=== Salary ranking within department ===')
sql = '''
SELECT
    name,
    department,
    salary,
    ROW_NUMBER()  OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
    RANK()        OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
    DENSE_RANK()  OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank
FROM employees
ORDER BY department, salary DESC
'''
print(q(sql))

In [ ]:
# Running total and cumulative % of payroll
print('=== Running salary total (by hire date) ===')
sql = '''
SELECT
    name,
    hire_date,
    salary,
    SUM(salary) OVER (ORDER BY hire_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
        AS running_total
FROM employees
ORDER BY hire_date
'''
print(q(sql))

In [ ]:
# LAG and LEAD
print('=== Salary change relative to previous hire ===')
sql = '''
SELECT
    name,
    hire_date,
    salary,
    LAG(salary)  OVER (ORDER BY hire_date) AS prev_hire_salary,
    LEAD(salary) OVER (ORDER BY hire_date) AS next_hire_salary,
    salary - LAG(salary) OVER (ORDER BY hire_date) AS salary_delta
FROM employees
ORDER BY hire_date
'''
print(q(sql))

In [ ]:
# NTILE — salary quartiles
print('=== Salary quartiles ===')
sql = '''
SELECT
    name,
    salary,
    NTILE(4) OVER (ORDER BY salary) AS quartile
FROM employees
ORDER BY salary
'''
print(q(sql))

## Exercises

1. Rank employees globally by salary (no partitioning), using `DENSE_RANK`.
2. For each employee, show their salary and the highest salary in their department using `MAX() OVER`.
3. Calculate a 3-employee moving average of salary ordered by `hire_date`.
4. Show each employee's salary percentile within the company using `PERCENT_RANK()`.

## Mini Challenge

Find the top-earning employee in each department. Use a window function and a CTE — do not use subqueries in WHERE.

## Best Practices

- Use CTEs to isolate window function results before filtering.
- `ROWS BETWEEN` is more precise than `RANGE BETWEEN` for numeric windows.
- Window functions cannot appear in WHERE — wrap in a CTE or subquery.

## Common Mistakes

- Filtering on a window function directly in WHERE — SQL does not allow this.
- Confusing `RANK` (gaps) vs `DENSE_RANK` (no gaps) — choose based on requirement.
- `LAST_VALUE` without `ROWS BETWEEN ... UNBOUNDED FOLLOWING` returns unexpected results.

## Summary

- Window functions operate over a window of rows without grouping the result.
- `PARTITION BY` divides the window; `ORDER BY` defines order within.
- Ranking, running totals, LAG/LEAD, and moving averages are core patterns.

---


### Exercise and Challenge Solutions — Section 7


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3: 3-employee moving average
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4: PERCENT_RANK
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: top earner per department via CTE
# YOUR SQL HERE
print(q(''))

# Section 8 — Data Manipulation Language (DML)

## Concept

DML statements modify data: `INSERT`, `UPDATE`, `DELETE`.
Transactions ensure atomicity: either all changes commit or all roll back.

## Technical Deep Dive

**INSERT:**
```sql
INSERT INTO table (col1, col2) VALUES (val1, val2);
INSERT INTO table SELECT ...;  -- insert from query
INSERT OR REPLACE INTO table ...;  -- upsert (SQLite)
```

**UPDATE:**
```sql
UPDATE table SET col = value WHERE condition;
```

**DELETE:**
```sql
DELETE FROM table WHERE condition;
-- No WHERE = delete all rows (dangerous!)
```

**Transactions:**
```sql
BEGIN;   -- or BEGIN TRANSACTION
-- DML statements
COMMIT;  -- or ROLLBACK;
```

**ACID properties:** Atomicity, Consistency, Isolation, Durability.


In [ ]:
# INSERT single row
cursor.execute('''
    INSERT INTO employees (id, name, department, salary, hire_date, manager_id)
    VALUES (11, 'Lena Park', 'Finance', 77000, '2023-01-20', NULL)
''')
conn.commit()
print('After INSERT:')
print(q("SELECT * FROM employees WHERE department = 'Finance'"))

In [ ]:
# UPDATE — 10% raise for Engineering
cursor.execute("UPDATE employees SET salary = ROUND(salary * 1.10, 0) WHERE department = 'Engineering'")
conn.commit()
print('After 10% Engineering raise:')
print(q("SELECT name, salary FROM employees WHERE department = 'Engineering'"))

In [ ]:
# Transaction with ROLLBACK example
print('Before transaction:')
print(q("SELECT COUNT(*) AS cnt FROM employees"))

try:
    cursor.execute('BEGIN')
    cursor.execute("INSERT INTO employees (id, name, department, salary, hire_date) VALUES (99, 'Test User', 'HR', 50000, '2024-01-01')")
    # Simulate an error condition
    raise ValueError('Something went wrong!')
    conn.commit()
except ValueError as e:
    conn.rollback()
    print(f'Rolled back: {e}')

print('After rollback:')
print(q("SELECT COUNT(*) AS cnt FROM employees"))

In [ ]:
# UPSERT — INSERT OR REPLACE (SQLite)
# Standard SQL: INSERT ... ON CONFLICT DO UPDATE
cursor.execute('''
    INSERT INTO departments (id, name, budget, location)
    VALUES (5, 'Finance', 300000, 'New York')
    ON CONFLICT(id) DO UPDATE SET budget = excluded.budget, location = excluded.location
''')
conn.commit()
print('After UPSERT on Finance department:')
print(q("SELECT * FROM departments WHERE id = 5"))

## Exercises

1. Insert a new department called 'Legal' with budget 100000 in 'Boston'.
2. Give all employees hired before 2020 a 5% salary increase.
3. Delete the employee with `id = 11` (Lena Park we just inserted).
4. Write a transaction that transfers 50000 budget from 'Marketing' to 'HR'. Rollback if either update fails.

## Mini Challenge

Write a Python function `safe_update(conn, department, raise_pct)` that applies a salary raise within a transaction and rolls back on any error, returning `(success: bool, affected_rows: int)`.

## Best Practices

- Always use `WHERE` in `UPDATE` and `DELETE` — review it twice before running.
- Use parameterized queries (never string interpolation) to prevent SQL injection.
- Wrap multi-step DML in transactions for atomicity.

## Common Mistakes

- `DELETE FROM table` without `WHERE` deletes all rows.
- Forgetting to `COMMIT` — changes are invisible to other connections.
- Using string formatting in SQL (`f'WHERE id = {user_input}'`) — SQL injection risk.

## Summary

- `INSERT`, `UPDATE`, `DELETE` modify data.
- Transactions guarantee ACID properties.
- Use parameterized queries: `cursor.execute(sql, (param,))`.
- UPSERT: `INSERT ... ON CONFLICT DO UPDATE`.

---


### Exercise and Challenge Solutions — Section 8


In [ ]:
# Exercise 1
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 2 — 5% raise for pre-2020 hires
# YOUR CODE HERE

In [ ]:
# Exercise 3
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 4 — budget transfer transaction
def transfer_budget(conn, from_dept, to_dept, amount):
    # YOUR CODE HERE
    pass

In [ ]:
# Mini Challenge: safe_update function
def safe_update(conn, department, raise_pct):
    # YOUR CODE HERE
    pass

# Section 9 — DDL and Schema Design

## Concept

DDL (Data Definition Language) defines the structure of the database:
tables, columns, constraints, indexes, and views.

## Technical Deep Dive

**Column constraints:**

| Constraint | Effect |
|---|---|
| `PRIMARY KEY` | Unique + NOT NULL, identifies each row |
| `NOT NULL` | Rejects NULL values |
| `UNIQUE` | Rejects duplicate values |
| `CHECK (expr)` | Rejects rows where expr is false |
| `DEFAULT value` | Sets default when not provided |
| `REFERENCES table(col)` | Foreign key constraint |

**Indexes:**
- Speed up reads on `WHERE`, `ORDER BY`, `JOIN` columns.
- Slow down writes (index must be updated on insert/update/delete).
- `CREATE INDEX idx_name ON table(col);`
- `CREATE UNIQUE INDEX ...` enforces uniqueness.

**Views:**
- Virtual table defined by a query.
- `CREATE VIEW v AS SELECT ...`
- Simplify complex queries; no data duplication.


In [ ]:
# CREATE TABLE with constraints
cursor.executescript('''
CREATE TABLE IF NOT EXISTS products (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    sku         TEXT    NOT NULL UNIQUE,
    name        TEXT    NOT NULL,
    price       REAL    NOT NULL CHECK (price >= 0),
    stock       INTEGER NOT NULL DEFAULT 0 CHECK (stock >= 0),
    category    TEXT,
    created_at  TEXT    DEFAULT (date('now'))
);

INSERT INTO products (sku, name, price, stock, category) VALUES
    ('SKU001', 'Laptop',     1299.99, 50, 'Electronics'),
    ('SKU002', 'Mouse',        29.99, 200,'Electronics'),
    ('SKU003', 'Desk Chair',  399.99, 30, 'Furniture'),
    ('SKU004', 'Monitor',     499.99, 75, 'Electronics'),
    ('SKU005', 'Keyboard',     79.99, 150,'Electronics');
''')
conn.commit()
print(q('SELECT * FROM products'))

In [ ]:
# Create indexes
cursor.executescript('''
CREATE INDEX IF NOT EXISTS idx_products_category ON products(category);
CREATE INDEX IF NOT EXISTS idx_employees_dept    ON employees(department);
CREATE INDEX IF NOT EXISTS idx_employees_salary  ON employees(salary);
''')
conn.commit()

# Show all indexes
print('=== All indexes ===')
print(q("SELECT name, tbl_name FROM sqlite_master WHERE type='index' ORDER BY tbl_name, name"))

In [ ]:
# Create a view
cursor.execute('''
CREATE VIEW IF NOT EXISTS v_employee_summary AS
SELECT
    e.name,
    e.department,
    e.salary,
    e.hire_date,
    ROUND(AVG(e2.salary) OVER (PARTITION BY e.department), 0) AS dept_avg_salary,
    COUNT(ep.project_id) OVER (PARTITION BY e.id)            AS project_count
FROM employees e
LEFT JOIN employee_projects ep ON e.id = ep.employee_id
''')
conn.commit()
print('=== Employee summary view ===')
print(q('SELECT DISTINCT name, department, salary, dept_avg_salary, project_count FROM v_employee_summary ORDER BY department, salary DESC'))

In [ ]:
# EXPLAIN QUERY PLAN — see how SQLite executes a query
print('=== Query plan WITHOUT index hint ===')
for row in cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM employees WHERE salary > 80000"):
    print(dict(row))

print('\n=== Query plan WITH index on salary ===')
for row in cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM employees WHERE department = 'Engineering'"):
    print(dict(row))

## Exercises

1. Create a `customers` table with: `id`, `email` (unique), `full_name`, `country`, `created_at` (default today).
2. Add a `CHECK` constraint to ensure `price > 0` on `products` (add it via a new column or recreate).
3. Create an index on `products.category`.
4. Create a view `v_expensive_products` showing products with `price > 200`.

## Mini Challenge

Design a schema for an e-commerce order system: `customers`, `orders`, `order_items`, `products`. Include primary keys, foreign keys, and appropriate constraints. Create the tables and insert 3 sample orders.

## Best Practices

- Always define `NOT NULL` where data must always be present.
- Use `CHECK` constraints to enforce business rules at the database level.
- Index foreign key columns — they are often used in JOINs.
- Use `CREATE INDEX IF NOT EXISTS` to avoid errors on re-runs.

## Common Mistakes

- Over-indexing: too many indexes slow down writes significantly.
- Using `TEXT` for all data types — prevents numeric comparisons and sorting.
- Forgetting to index foreign key columns — leads to full table scans on JOINs.

## Summary

- DDL defines tables, constraints, indexes, and views.
- Constraints enforce data integrity at the database level.
- Indexes trade write speed for read speed.
- Views simplify complex queries without data duplication.

---


### Exercise and Challenge Solutions — Section 9


In [ ]:
# Exercise 1
# YOUR CODE HERE

In [ ]:
# Exercise 3 (index on products.category already created above, here for standalone)
# YOUR CODE HERE

In [ ]:
# Exercise 4
# YOUR SQL HERE
print(q(''))

In [ ]:
# Mini Challenge: e-commerce schema
# YOUR SQL HERE
print(q(''))

# Section 10 — Python SQL Integration

## Concept

Python integrates with SQL databases via:
- **sqlite3** — standard library, no install needed
- **pandas** — `read_sql_query` / `to_sql`
- **SQLAlchemy** — ORM and connection pooling for production
- **psycopg2** / **asyncpg** — PostgreSQL drivers

## Technical Deep Dive

**Parameterized queries — always use them:**
```python
# WRONG — SQL injection risk
cursor.execute(f"SELECT * FROM users WHERE name = '{name}'")

# CORRECT
cursor.execute('SELECT * FROM users WHERE name = ?', (name,))
```

**sqlite3 row_factory options:**
```python
conn.row_factory = sqlite3.Row       # access by name: row['col']
conn.row_factory = lambda c, r: dict(zip([d[0] for d in c.description], r))  # dict
```

**Context manager:**
```python
with sqlite3.connect('db.sqlite') as conn:
    conn.execute('INSERT ...')
    # auto-commits on exit, rolls back on exception
```


In [ ]:
# Parameterized queries — safe pattern
def find_employees_by_dept(conn, department):
    cursor = conn.cursor()
    cursor.execute(
        'SELECT name, salary FROM employees WHERE department = ? ORDER BY salary DESC',
        (department,)
    )
    return cursor.fetchall()

rows = find_employees_by_dept(conn, 'Engineering')
for row in rows:
    print(f'{row["name"]}: ${row["salary"]:,.0f}')

In [ ]:
# pandas integration
df = pd.read_sql_query('SELECT * FROM employees', conn)
print('DataFrame shape:', df.shape)
print(df.dtypes)
print(df.head())

# Analysis with pandas after loading
print('\nSalary stats by department:')
print(df.groupby('department')['salary'].agg(['mean', 'min', 'max', 'count']).round(0))

In [ ]:
# Write DataFrame to SQL
salary_summary = df.groupby('department').agg(
    avg_salary=('salary', 'mean'),
    headcount=('name', 'count')
).reset_index()
salary_summary['avg_salary'] = salary_summary['avg_salary'].round(0)

salary_summary.to_sql('dept_salary_summary', conn, if_exists='replace', index=False)
print('Written to SQL:')
print(q('SELECT * FROM dept_salary_summary ORDER BY avg_salary DESC'))

In [ ]:
# executemany — bulk insert
new_employees = [
    ('Mark Evans',   'Legal', 72000, '2024-03-01', None),
    ('Sara Kim',     'Legal', 68000, '2024-03-15', None),
    ('Tom Baker',    'Legal', 75000, '2024-04-01', None),
]

cursor.executemany(
    'INSERT INTO employees (name, department, salary, hire_date, manager_id) VALUES (?, ?, ?, ?, ?)',
    new_employees
)
conn.commit()
print('After bulk insert:')
print(q("SELECT * FROM employees WHERE department = 'Legal'"))

In [ ]:
# SQLAlchemy Core (no ORM) — production-grade pattern
try:
    from sqlalchemy import create_engine, text

    engine = create_engine('sqlite:///:memory:', echo=False)

    # Create table and insert using SQLAlchemy
    with engine.connect() as sa_conn:
        sa_conn.execute(text('''
            CREATE TABLE IF NOT EXISTS logs (
                id      INTEGER PRIMARY KEY,
                event   TEXT,
                ts      TEXT DEFAULT (datetime('now'))
            )
        '''))
        sa_conn.execute(text("INSERT INTO logs (event) VALUES ('app_start'), ('user_login')"))
        sa_conn.commit()

        result = sa_conn.execute(text('SELECT * FROM logs'))
        for row in result:
            print(row)

    print('SQLAlchemy integration successful.')
except ImportError:
    print('SQLAlchemy not installed. Run: pip install sqlalchemy')
    print('The concept: SQLAlchemy wraps connections, provides connection pooling and ORM.')

In [ ]:
# Context manager with file-based SQLite
import tempfile
import os

tmp = tempfile.mktemp(suffix='.db')
try:
    with sqlite3.connect(tmp) as file_conn:
        file_conn.execute('CREATE TABLE test (id INTEGER PRIMARY KEY, val TEXT)')
        file_conn.execute("INSERT INTO test VALUES (1, 'hello'), (2, 'world')")
        # auto-commits on clean exit

    # Re-open and verify persistence
    with sqlite3.connect(tmp) as file_conn:
        rows = file_conn.execute('SELECT * FROM test').fetchall()
        print('Persisted rows:', rows)
finally:
    os.unlink(tmp)

## Exercises

1. Write a function `get_dept_report(conn, dept)` that returns a dictionary with `avg_salary`, `headcount`, and `names` (list).
2. Using `executemany`, insert 5 products into the `products` table from a list of tuples.
3. Load the `employees` table into a pandas DataFrame and compute the correlation between `salary` and years employed.
4. Write a function `search_employees(conn, name_fragment, min_salary)` using parameterized queries.

## Mini Challenge

Build a small `DatabaseManager` class with:
- `__init__(db_path)` — opens connection, sets `row_factory`
- `query(sql, params)` → list of dicts
- `execute(sql, params)` → rowcount (wraps in transaction)
- `close()`
- Context manager support (`__enter__` / `__exit__`)

## Best Practices

- Always use parameterized queries — never string interpolation in SQL.
- Use connection pooling (SQLAlchemy) in web applications.
- Close connections explicitly or use context managers.
- Use `pd.read_sql_query` for analysis; avoid loading entire tables into memory.

## Common Mistakes

- SQL injection via f-strings — never do this with user input.
- Not closing connections — leads to database locks.
- Using `fetchall()` on large tables — loads all rows into memory.
- Ignoring exceptions after `BEGIN` without `ROLLBACK` — leaves transactions open.

## Summary

- `sqlite3`: built-in, file or in-memory, good for learning and small apps.
- `pandas.read_sql_query` integrates SQL with data analysis.
- `executemany` for bulk inserts.
- SQLAlchemy for production: connection pooling, ORM, migrations.
- Always use parameterized queries.

---


### Exercise and Challenge Solutions — Section 10


In [ ]:
# Exercise 1
def get_dept_report(conn, dept):
    # YOUR CODE HERE
    pass

In [ ]:
# Exercise 2 — bulk insert products
# YOUR SQL HERE
print(q(''))

In [ ]:
# Exercise 3 — salary vs years employed correlation
# YOUR CODE HERE

In [ ]:
# Exercise 4
def search_employees(conn, name_fragment, min_salary):
    # YOUR CODE HERE
    pass

In [ ]:
# Mini Challenge: DatabaseManager class
class DatabaseManager:
    # YOUR CODE HERE
    pass

# Course Summary

| Section | Key Skills |
|---|---|
| 1. Intro | Relational model, SQLite setup, sample DB |
| 2. SELECT | Columns, aliases, DISTINCT, LIMIT, OFFSET |
| 3. Filtering | WHERE, BETWEEN, IN, LIKE, IS NULL, CASE |
| 4. Aggregation | COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING |
| 5. JOINs | INNER, LEFT, SELF, anti-join, multi-join |
| 6. Subqueries & CTEs | Scalar, correlated, EXISTS, WITH, recursive |
| 7. Window Functions | RANK, ROW_NUMBER, LAG/LEAD, running totals |
| 8. DML | INSERT, UPDATE, DELETE, transactions, upsert |
| 9. DDL | Tables, constraints, indexes, views |
| 10. Python Integration | sqlite3, pandas, executemany, SQLAlchemy |

## Next Steps

- **PostgreSQL** — window functions, JSONB, full-text search, EXPLAIN ANALYZE
- **dbt** — data transformation in SQL at scale
- **SQLAlchemy ORM** — Python class → table mapping
- **Pandas + SQL** — EDA with `numpy_pandas_course.ipynb`

---
